# B3 — ResNet50 + Vanilla EDL + KL

Google Colab / Google Drive pipeline.

Primary B3:
- ResNet50 backbone
- Softplus evidence head
- alpha = evidence + C
- C = 1.0 default
- expected Dirichlet cross-entropy
- KL regularization
- Grad-CAM + ESM
- 5-fold patient-level CV
- Brier early stopping
- F1-Macro, AUROC, Brier, ECE, NLL
- vacuity rejection curve
- latency, FLOPs
- fold-0 ablation hooks for prior C, loss isolation and saliency target

Current pilot mode uses at most 20 patients.

In [ ]:
!pip -q install "torch>=2.2" "torchvision>=0.17" scikit-learn pandas numpy matplotlib pillow tqdm thop scipy

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os, gc, json, math, time, random, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, StratifiedShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import label_binarize
from scipy.stats import wilcoxon
from tqdm.auto import tqdm
from thop import profile

warnings.filterwarnings("ignore")

In [ ]:
# Google Drive project layout
# Individual patch images only. This notebook does NOT read or require any .npz archive.
PROJECT_ROOT = Path("/content/drive/MyDrive/LIDC_Thesis")

# Primary locations produced by the preprocessing project.
PATCH_ROOT = PROJECT_ROOT / "patches"
METADATA_ROOT = PROJECT_ROOT / "metadata"

# Backward-compatible alternate locations from older notebook layouts.
PATCH_ROOT_CANDIDATES = [
    PATCH_ROOT,
    PROJECT_ROOT / "LIDC" / "processed" / "patches",
    PROJECT_ROOT / "processed" / "patches",
    PROJECT_ROOT / "data" / "processed" / "patches",
]

METADATA_CSV_CANDIDATES = [
    METADATA_ROOT / "nodule_metadata_filtered.csv",
    METADATA_ROOT / "lidc_patches_metadata.csv",
    PROJECT_ROOT / "patch_manifest.csv",
    PROJECT_ROOT / "LIDC" / "processed" / "metadata" / "lidc_patches_metadata.csv",
    PROJECT_ROOT / "processed" / "metadata" / "lidc_patches_metadata.csv",
]

# B3 experiment outputs.
B3_ROOT = PROJECT_ROOT / "experiments" / "B3_ResNet50_Vanilla_EDL_KL"
CONFIG_ROOT = B3_ROOT / "configs"
SPLIT_ROOT = B3_ROOT / "splits"
CHECKPOINT_ROOT = B3_ROOT / "checkpoints"
PREDICTION_ROOT = B3_ROOT / "predictions"
METRICS_ROOT = B3_ROOT / "metrics"
GRADCAM_ROOT = B3_ROOT / "gradcam"
ESM_ROOT = B3_ROOT / "esm"
PLOT_ROOT = B3_ROOT / "plots"
LOG_ROOT = B3_ROOT / "logs"
REPORT_ROOT = B3_ROOT / "reports"
ABLATION_ROOT = B3_ROOT / "ablations"

for p in [
    METADATA_ROOT, CONFIG_ROOT, SPLIT_ROOT, CHECKPOINT_ROOT, PREDICTION_ROOT,
    METRICS_ROOT, GRADCAM_ROOT, ESM_ROOT, PLOT_ROOT, LOG_ROOT,
    REPORT_ROOT, ABLATION_ROOT
]:
    p.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def choose_patch_root():
    """Pick the first candidate containing at least one individual image."""
    for root in PATCH_ROOT_CANDIDATES:
        if root.exists():
            try:
                if any(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in root.rglob("*")):
                    return root
            except Exception:
                pass
    return PATCH_ROOT

PATCH_ROOT = choose_patch_root()
print("PROJECT_ROOT :", PROJECT_ROOT)
print("PATCH_ROOT   :", PATCH_ROOT)
print("METADATA_ROOT:", METADATA_ROOT)


PROJECT_ROOT : /content/drive/MyDrive/LIDC_Thesis
PATCH_ROOT   : /content/drive/MyDrive/LIDC_Thesis/LIDC/processed/patches
METADATA_ROOT: /content/drive/MyDrive/LIDC_Thesis/metadata


In [ ]:
CONFIG = {
    "config_id":"B3",
    "backbone":"ResNet50",
    "uq_method":"Vanilla EDL",
    "num_classes":3,
    "class_names":["benign","indeterminate","malignant"],
    "seed":42,
    "n_splits":5,
    "validation_fraction_of_training_patients":0.10,
    "image_size":224,
    "batch_size":32,
    "num_workers":0,  # Colab-safe: disable multiprocessing workers
    "max_epochs":50,
    "early_stopping_patience":10,
    "learning_rate":1e-4,
    "eta_min":1e-6,
    "weight_decay":0.01,
    "adam_beta1":0.9,
    "adam_beta2":0.999,
    "prior_weight_C":1.0,
    "lambda_kl":1.0,
    "epsilon":1e-8,
    "ece_bins":15,
    "random_rejection_repeats":50,
    "gradcam_max_examples_per_fold":30,
    "esm_max_examples_per_fold":30,
    "xai_metric_steps":20,
    "pilot_max_patients":50,
    "ablation_prior_C_values":[0.5,1.0,2.0],
    "ablation_loss_modes":["ece_only","ece_plus_kl"],
}

with open(CONFIG_ROOT/"B3_config.json","w") as f:
    json.dump(CONFIG,f,indent=2)

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"]=str(seed)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

seed_everything(CONFIG["seed"])
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",DEVICE)

Device: cuda


In [ ]:
import re

LABEL_MAP = {"benign": 0, "indeterminate": 1, "malignant": 2}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

def normalize_patient_id(value):
    """Normalize patient IDs while preserving the standard LIDC-IDRI-#### form."""
    s = str(value).strip()
    m = re.search(r"(LIDC-IDRI-\d{4})", s, flags=re.IGNORECASE)
    return m.group(1).upper() if m else s

def infer_patient_id_from_path(fp):
    """Infer patient ID from any directory component or filename."""
    for token in list(fp.parts)[::-1]:
        m = re.search(r"(LIDC-IDRI-\d{4})", str(token), flags=re.IGNORECASE)
        if m:
            return m.group(1).upper()
    return fp.stem.split("_")[0]

def infer_class_from_path(fp):
    """Infer ternary class from folder names or filename text."""
    tokens = [str(x).lower() for x in fp.parts]
    stem = fp.stem.lower()
    for cname, label in LABEL_MAP.items():
        if cname in tokens or re.search(rf"(^|[_\-.]){re.escape(cname)}([_\-.]|$)", stem):
            return cname, label
    return None, None

def discover_patch_images(root):
    """Return all supported individual patch image files recursively."""
    if not root.exists():
        return []
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

def find_metadata_csv():
    """Use the first known metadata CSV that exists."""
    for fp in METADATA_CSV_CANDIDATES:
        if fp.exists():
            return fp
    return None

def standardize_metadata_columns(df):
    """Map common preprocessing column names to the B3 names."""
    df = df.copy()

    aliases = {
        "patient_id": ["patient_id", "patient", "patient_uid", "case_id", "subject_id"],
        "label": ["label", "class_id", "target", "y"],
        "class_name": ["class_name", "label_name", "class", "category"],
        "image_path": ["image_path", "patch_path", "filepath", "file_path", "path"],
        "nodule_id": ["nodule_id", "nodule_key", "lesion_id", "annotation_id"],
    }

    lower_to_original = {str(c).lower(): c for c in df.columns}
    for target, candidates in aliases.items():
        if target in df.columns:
            continue
        for candidate in candidates:
            if candidate.lower() in lower_to_original:
                df[target] = df[lower_to_original[candidate.lower()]]
                break

    # If only class-name text exists, convert it to numeric labels.
    if "label" not in df.columns and "class_name" in df.columns:
        df["class_name"] = df["class_name"].astype(str).str.strip().str.lower()
        df["label"] = df["class_name"].map(LABEL_MAP)

    # If only numeric labels exist, add class names.
    if "class_name" not in df.columns and "label" in df.columns:
        numeric = pd.to_numeric(df["label"], errors="coerce")
        df["class_name"] = numeric.map(INV_LABEL_MAP)

    return df

def build_image_index(image_files):
    """
    Build lookup tables for individual patch files.
    Matching preference later is:
      1) exact resolved path
      2) filename
      3) filename stem / nodule_id
    """
    by_name = {}
    by_stem = {}
    by_resolved = {}

    for fp in image_files:
        by_name.setdefault(fp.name, []).append(fp)
        by_stem.setdefault(fp.stem, []).append(fp)
        try:
            by_resolved[str(fp.resolve())] = fp
        except Exception:
            by_resolved[str(fp)] = fp

    return by_name, by_stem, by_resolved

def match_image_for_metadata_row(row, by_name, by_stem, by_resolved):
    """Resolve a metadata record to an actual individual patch image."""
    # 1. Existing explicit path.
    if "image_path" in row.index and pd.notna(row["image_path"]):
        raw = str(row["image_path"]).strip()
        if raw:
            p = Path(raw)
            if p.exists() and p.suffix.lower() in IMAGE_EXTENSIONS:
                return p
            try:
                key = str(p.resolve())
                if key in by_resolved:
                    return by_resolved[key]
            except Exception:
                pass
            # Old absolute paths may no longer be valid after moving Drive folders.
            if p.name in by_name and len(by_name[p.name]) == 1:
                return by_name[p.name][0]
            if p.stem in by_stem and len(by_stem[p.stem]) == 1:
                return by_stem[p.stem][0]

    # 2. Match the nodule identifier to an image stem.
    if "nodule_id" in row.index and pd.notna(row["nodule_id"]):
        nid = str(row["nodule_id"]).strip()
        candidates = [
            nid,
            nid.replace("/", "_"),
            nid.replace(" ", "_"),
        ]
        for key in candidates:
            if key in by_stem and len(by_stem[key]) == 1:
                return by_stem[key][0]

        # Prefix/substring fallback for filenames such as "<nodule_id>_patch.png".
        fuzzy = []
        for stem, paths in by_stem.items():
            if stem == nid or stem.startswith(nid + "_") or nid in stem:
                fuzzy.extend(paths)
        if len(fuzzy) == 1:
            return fuzzy[0]

    return None

def metadata_from_individual_patches(root):
    """
    Build the training dataframe from individual patch images.
    Works with either:
      A) class folders:
         patches/benign/...png
         patches/indeterminate/...png
         patches/malignant/...png
      B) a flat/arbitrary patch folder plus preprocessing metadata such as
         metadata/nodule_metadata_filtered.csv containing nodule_id/patient_id/label.
    """
    image_files = discover_patch_images(root)

    if not image_files:
        searched = "\n".join(f"  - {p}" for p in PATCH_ROOT_CANDIDATES)
        raise RuntimeError(
            "No individual patch image files were found.\n"
            "Expected PNG/JPG/TIF/BMP patch files under one of:\n"
            f"{searched}\n"
            "This B3 notebook intentionally does not use patches_224.npz."
        )

    print(f"Found {len(image_files)} individual patch image files under: {root}")

    by_name, by_stem, by_resolved = build_image_index(image_files)
    metadata_csv = find_metadata_csv()
    rows = []

    # Prefer metadata because it preserves the preprocessing labels/patient grouping.
    if metadata_csv is not None:
        print("Using metadata:", metadata_csv)
        src = standardize_metadata_columns(pd.read_csv(metadata_csv))

        matched = 0
        for _, r in src.iterrows():
            fp = match_image_for_metadata_row(r, by_name, by_stem, by_resolved)
            if fp is None:
                continue

            # Label: metadata first, class-folder inference second.
            label = None
            class_name = None

            if "label" in src.columns and pd.notna(r.get("label", np.nan)):
                try:
                    label = int(float(r["label"]))
                except Exception:
                    label = None

            if "class_name" in src.columns and pd.notna(r.get("class_name", np.nan)):
                class_name = str(r["class_name"]).strip().lower()
                if label is None:
                    label = LABEL_MAP.get(class_name)

            if label not in INV_LABEL_MAP:
                inferred_name, inferred_label = infer_class_from_path(fp)
                class_name, label = inferred_name, inferred_label

            if label not in INV_LABEL_MAP:
                continue
            if class_name not in LABEL_MAP:
                class_name = INV_LABEL_MAP[int(label)]

            # Patient ID: metadata first, then path inference.
            if "patient_id" in src.columns and pd.notna(r.get("patient_id", np.nan)):
                patient_id = normalize_patient_id(r["patient_id"])
            else:
                patient_id = infer_patient_id_from_path(fp)

            row = {
                "image_path": str(fp),
                "patient_id": str(patient_id),
                "label": int(label),
                "class_name": class_name,
            }

            # Keep useful optional columns if present.
            for c in ["nodule_id", "V", "variance", "alignment_mask", "fold", "cv_fold", "mask_path"]:
                if c in src.columns:
                    row[c] = r[c]

            rows.append(row)
            matched += 1

        print(f"Matched metadata to {matched} individual patch images.")

    # If metadata is absent/unusable, construct records directly from class folders/filenames.
    if not rows:
        print("Metadata could not be used; inferring labels directly from individual patch paths.")
        for fp in image_files:
            class_name, label = infer_class_from_path(fp)
            if label is None:
                continue
            rows.append({
                "image_path": str(fp),
                "patient_id": infer_patient_id_from_path(fp),
                "label": int(label),
                "class_name": class_name,
            })

    out = pd.DataFrame(rows)

    if out.empty:
        raise RuntimeError(
            "Individual patch images were found, but labels could not be determined.\n"
            "Use class folders named benign/indeterminate/malignant, OR provide "
            "metadata/nodule_metadata_filtered.csv with label (or label_name) and nodule_id/image_path."
        )

    out["image_path"] = out["image_path"].astype(str)
    out["patient_id"] = out["patient_id"].astype(str).map(normalize_patient_id)
    out["label"] = pd.to_numeric(out["label"], errors="coerce")
    out = out[out["label"].isin([0, 1, 2])].copy()
    out["label"] = out["label"].astype(int)
    out["class_name"] = out["label"].map(INV_LABEL_MAP)

    # Final existence check and de-duplication.
    out = out[out["image_path"].map(lambda p: Path(p).is_file())]
    out = out.drop_duplicates("image_path").reset_index(drop=True)
    return out

metadata = metadata_from_individual_patches(PATCH_ROOT)

if CONFIG["pilot_max_patients"] is not None:
    selected = sorted(metadata["patient_id"].unique())[:CONFIG["pilot_max_patients"]]
    metadata = metadata[metadata["patient_id"].isin(selected)].reset_index(drop=True)

if metadata.empty:
    raise RuntimeError("No usable individual patch records remain after filtering.")

metadata.to_csv(METADATA_ROOT / "B3_effective_metadata.csv", index=False)

print("\nB3 individual-patch dataset ready")
print("Patches :", len(metadata))
print("Patients:", metadata["patient_id"].nunique())
print("\nClass distribution:")
display(metadata["class_name"].value_counts().rename_axis("class").to_frame("patches"))
display(metadata.head())


Found 94 individual patch image files under: /content/drive/MyDrive/LIDC_Thesis/LIDC/processed/patches
Using metadata: /content/drive/MyDrive/LIDC_Thesis/LIDC/processed/metadata/lidc_patches_metadata.csv
Matched metadata to 0 individual patch images.
Metadata could not be used; inferring labels directly from individual patch paths.

B3 individual-patch dataset ready
Patches : 94
Patients: 38

Class distribution:


,patches
class,
indeterminate,54
malignant,21
benign,19


,image_path,patient_id,label,class_name
0,/content/drive/MyDrive/LIDC_Thesis/LIDC/proces...,LIDC-IDRI-0004,0,benign
1,/content/drive/MyDrive/LIDC_Thesis/LIDC/proces...,LIDC-IDRI-0011,0,benign
2,/content/drive/MyDrive/LIDC_Thesis/LIDC/proces...,LIDC-IDRI-0018,0,benign
3,/content/drive/MyDrive/LIDC_Thesis/LIDC/proces...,LIDC-IDRI-0021,0,benign
4,/content/drive/MyDrive/LIDC_Thesis/LIDC/proces...,LIDC-IDRI-0024,0,benign


In [ ]:
bad=[]
for p in tqdm(metadata["image_path"],desc="Checking images"):
    try:
        with Image.open(p) as img:
            img.verify()
    except Exception as e:
        bad.append({"image_path":p,"error":repr(e)})

if bad:
    pd.DataFrame(bad).to_csv(LOG_ROOT/"unreadable_images.csv",index=False)
    badset={x["image_path"] for x in bad}
    metadata=metadata[~metadata["image_path"].isin(badset)].reset_index(drop=True)

if metadata["patient_id"].nunique()<CONFIG["n_splits"]:
    raise RuntimeError("Not enough patients for 5-fold CV.")

Checking images:   0%|          | 0/94 [00:00<?, ?it/s]

In [ ]:
def dominant_patient_label(df):
    return df.groupby("patient_id")["label"].agg(lambda s:int(s.value_counts().index[0]))

def make_outer_splits(df):
    X=np.zeros((len(df),1)); y=df["label"].values; g=df["patient_id"].values
    try:
        sp=StratifiedGroupKFold(n_splits=CONFIG["n_splits"],shuffle=True,random_state=CONFIG["seed"])
        return list(sp.split(X,y,g)),"StratifiedGroupKFold"
    except Exception:
        sp=GroupKFold(n_splits=CONFIG["n_splits"])
        return list(sp.split(X,y,g)),"GroupKFold"

def make_train_val(trainval,fold):
    p=dominant_patient_label(trainval)
    patients=p.index.to_numpy(); labels=p.values
    nval=max(1,int(round(len(patients)*CONFIG["validation_fraction_of_training_patients"])))
    nval=min(nval,len(patients)-1)
    counts=Counter(labels)
    ok=len(counts)>=2 and min(counts.values())>=2 and nval>=len(counts)
    if ok:
        ss=StratifiedShuffleSplit(n_splits=1,test_size=nval,random_state=CONFIG["seed"]+fold)
        ti,vi=next(ss.split(patients,labels))
        trp,vap=set(patients[ti]),set(patients[vi])
        method="StratifiedShuffleSplit(patient)"
    else:
        rng=np.random.RandomState(CONFIG["seed"]+fold)
        q=patients.copy(); rng.shuffle(q)
        vap=set(q[:nval]); trp=set(q[nval:])
        method="RandomPatientSplit(fallback)"
    return (
        trainval[trainval.patient_id.isin(trp)].copy(),
        trainval[trainval.patient_id.isin(vap)].copy(),
        method,
    )

splits,outer_method=make_outer_splits(metadata)
fold_frames={}; manifest=[]

for fold,(tvi,tei) in enumerate(splits):
    tv=metadata.iloc[tvi].copy(); te=metadata.iloc[tei].copy()
    tr,va,val_method=make_train_val(tv,fold)

    assert set(tr.patient_id).isdisjoint(set(va.patient_id))
    assert set(tr.patient_id).isdisjoint(set(te.patient_id))
    assert set(va.patient_id).isdisjoint(set(te.patient_id))

    fold_frames[fold]={"train":tr.reset_index(drop=True),"val":va.reset_index(drop=True),"test":te.reset_index(drop=True)}

    for name,frame in fold_frames[fold].items():
        x=frame.copy(); x["fold"]=fold; x["split"]=name
        x.to_csv(SPLIT_ROOT/f"fold_{fold}_{name}.csv",index=False)
        manifest.append({
            "fold":fold,"split":name,"outer_method":outer_method,
            "validation_method":val_method if name!="test" else "",
            "patches":len(frame),"patients":frame.patient_id.nunique(),
            "benign":int((frame.label==0).sum()),
            "indeterminate":int((frame.label==1).sum()),
            "malignant":int((frame.label==2).sum()),
        })

split_manifest=pd.DataFrame(manifest)
split_manifest.to_csv(SPLIT_ROOT/"split_manifest.csv",index=False)
display(split_manifest)

print(manifest)

,fold,split,outer_method,validation_method,patches,patients,benign,indeterminate,malignant
0,0,train,StratifiedGroupKFold,StratifiedShuffleSplit(patient),66,29,10,39,17
1,0,val,StratifiedGroupKFold,StratifiedShuffleSplit(patient),9,3,4,4,1
2,0,test,StratifiedGroupKFold,,19,6,5,11,3
3,1,train,StratifiedGroupKFold,StratifiedShuffleSplit(patient),65,26,13,37,15
4,1,val,StratifiedGroupKFold,StratifiedShuffleSplit(patient),7,3,2,4,1
5,1,test,StratifiedGroupKFold,,22,9,4,13,5
6,2,train,StratifiedGroupKFold,StratifiedShuffleSplit(patient),56,26,12,29,15
7,2,val,StratifiedGroupKFold,StratifiedShuffleSplit(patient),15,3,0,13,2
8,2,test,StratifiedGroupKFold,,23,9,7,12,4
9,3,train,StratifiedGroupKFold,StratifiedShuffleSplit(patient),70,28,16,39,15


[{'fold': 0, 'split': 'train', 'outer_method': 'StratifiedGroupKFold', 'validation_method': 'StratifiedShuffleSplit(patient)', 'patches': 66, 'patients': 29, 'benign': 10, 'indeterminate': 39, 'malignant': 17}, {'fold': 0, 'split': 'val', 'outer_method': 'StratifiedGroupKFold', 'validation_method': 'StratifiedShuffleSplit(patient)', 'patches': 9, 'patients': 3, 'benign': 4, 'indeterminate': 4, 'malignant': 1}, {'fold': 0, 'split': 'test', 'outer_method': 'StratifiedGroupKFold', 'validation_method': '', 'patches': 19, 'patients': 6, 'benign': 5, 'indeterminate': 11, 'malignant': 3}, {'fold': 1, 'split': 'train', 'outer_method': 'StratifiedGroupKFold', 'validation_method': 'StratifiedShuffleSplit(patient)', 'patches': 65, 'patients': 26, 'benign': 13, 'indeterminate': 37, 'malignant': 15}, {'fold': 1, 'split': 'val', 'outer_method': 'StratifiedGroupKFold', 'validation_method': 'StratifiedShuffleSplit(patient)', 'patches': 7, 'patients': 3, 'benign': 2, 'indeterminate': 4, 'malignant': 1}

In [ ]:
IMAGENET_MEAN=[0.485,0.456,0.406]
IMAGENET_STD=[0.229,0.224,0.225]

base_transform=transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD),
])

class LIDCDataset(Dataset):
    def __init__(self,df):
        self.df=df.reset_index(drop=True).copy()
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        with Image.open(r.image_path) as img:
            x=base_transform(img.convert("L"))
        item={
            "image":x,
            "label":torch.tensor(int(r.label),dtype=torch.long),
            "image_path":str(r.image_path),
            "patient_id":str(r.patient_id),
        }
        if "mask_path" in self.df.columns:
            item["mask_path"]=str(r.mask_path) if pd.notna(r.mask_path) else ""
        return item

def make_loader(df,shuffle=False,batch_size=None):
    return DataLoader(
        LIDCDataset(df),
        batch_size=batch_size or CONFIG["batch_size"],
        shuffle=shuffle,
        num_workers=CONFIG["num_workers"],
        pin_memory=False,  # safer with num_workers=0 in Colab
        drop_last=False,
    )

In [ ]:
class B3ResNet50EDL(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=resnet50(weights=ResNet50_Weights.DEFAULT)
        self.backbone.fc=nn.Linear(self.backbone.fc.in_features,CONFIG["num_classes"])
    def forward(self,x):
        return self.backbone(x)

def build_model():
    return B3ResNet50EDL().to(DEVICE)

In [ ]:
EPS=CONFIG["epsilon"]

def logits_to_dirichlet(logits,prior_C=None):
    C=CONFIG["prior_weight_C"] if prior_C is None else float(prior_C)
    evidence=F.softplus(logits)
    alpha=evidence+C
    strength=alpha.sum(1,keepdim=True)
    probs=alpha/strength
    vacuity=(CONFIG["num_classes"]*C)/strength.squeeze(1)
    ent=-(probs*torch.log(probs.clamp_min(EPS))).sum(1)
    return {
        "evidence":evidence,
        "alpha":alpha,
        "strength":strength.squeeze(1),
        "probs":probs,
        "vacuity":vacuity,
        "aleatoric_entropy":ent,
        "normalized_aleatoric_entropy":ent/math.log(CONFIG["num_classes"]),
    }

def expected_dirichlet_ce(alpha,targets):
    y=F.one_hot(targets,num_classes=CONFIG["num_classes"]).float()
    S=alpha.sum(1,keepdim=True)
    return (y*(torch.digamma(S)-torch.digamma(alpha))).sum(1).mean()

def dirichlet_kl_to_uniform(alpha_tilde):
    beta=torch.ones_like(alpha_tilde)
    sa=alpha_tilde.sum(1,keepdim=True)
    sb=beta.sum(1,keepdim=True)
    logB_a=torch.lgamma(alpha_tilde).sum(1,keepdim=True)-torch.lgamma(sa)
    logB_b=torch.lgamma(beta).sum(1,keepdim=True)-torch.lgamma(sb)
    dig=((alpha_tilde-beta)*(torch.digamma(alpha_tilde)-torch.digamma(sa))).sum(1,keepdim=True)
    return (logB_b-logB_a+dig).mean()

def vanilla_edl_loss(logits,targets,prior_C=None,lambda_kl=None,loss_mode="ece_plus_kl"):
    C=CONFIG["prior_weight_C"] if prior_C is None else float(prior_C)
    lkl=CONFIG["lambda_kl"] if lambda_kl is None else float(lambda_kl)
    d=logits_to_dirichlet(logits,C)
    alpha=d["alpha"]
    lece=expected_dirichlet_ce(alpha,targets)
    y=F.one_hot(targets,num_classes=CONFIG["num_classes"]).float()
    alpha_tilde=y+(1-y)*alpha
    lkloss=dirichlet_kl_to_uniform(alpha_tilde)
    total=lece if loss_mode=="ece_only" else lece+lkl*lkloss
    return total,{"loss_total":float(total.detach()),"loss_ece":float(lece.detach()),"loss_kl":float(lkloss.detach())}

In [ ]:
def brier(y,p):
    yy=np.eye(3)[np.asarray(y,dtype=int)]
    return float(np.mean(np.sum((np.asarray(p)-yy)**2,1)))

def nll(y,p):
    y=np.asarray(y,dtype=int); p=np.clip(np.asarray(p),1e-12,1)
    return float(-np.mean(np.log(p[np.arange(len(y)),y])))

def ece(y,p,bins=15):
    y=np.asarray(y); p=np.asarray(p)
    conf=p.max(1); pred=p.argmax(1); corr=pred==y
    edges=np.linspace(0,1,bins+1); out=0.0
    for i in range(bins):
        lo,hi=edges[i],edges[i+1]
        m=(conf>=lo)&(conf<=hi) if i==0 else (conf>lo)&(conf<=hi)
        if m.any(): out+=m.mean()*abs(corr[m].mean()-conf[m].mean())
    return float(out)

def macro_auc(y,p):
    yb=label_binarize(np.asarray(y),classes=[0,1,2]); vals=[]
    for k in range(3):
        if len(np.unique(yb[:,k]))<2: vals.append(np.nan)
        else: vals.append(roc_auc_score(yb[:,k],np.asarray(p)[:,k]))
    vals=[v for v in vals if np.isfinite(v)]
    return float(np.mean(vals)) if vals else np.nan

def metrics(y,p):
    pred=np.asarray(p).argmax(1)
    return {
        "accuracy":float(accuracy_score(y,pred)),
        "f1_macro":float(f1_score(y,pred,average="macro",zero_division=0)),
        "auroc_macro_ovr":macro_auc(y,p),
        "brier_score":brier(y,p),
        "ece_15":ece(y,p,15),
        "nll":nll(y,p),
    }

In [ ]:
@torch.no_grad()
def edl_predict(model,loader,prior_C=None):
    model.eval()
    out={k:[] for k in ["y_true","probs","vacuity","aleatoric_entropy","normalized_aleatoric_entropy","strength","evidence"]}
    paths=[]; patients=[]
    for b in loader:
        x=b["image"].to(DEVICE)
        d=logits_to_dirichlet(model(x),prior_C)
        out["y_true"].append(b["label"].numpy())
        for k in ["probs","vacuity","aleatoric_entropy","normalized_aleatoric_entropy","strength","evidence"]:
            out[k].append(d[k].cpu().numpy())
        paths.extend(b["image_path"]); patients.extend(b["patient_id"])
    for k in out: out[k]=np.concatenate(out[k])
    out["paths"]=paths; out["patients"]=patients
    return out

In [ ]:
def train_one_fold(fold,train_df,val_df,prior_C=None,lambda_kl=None,loss_mode="ece_plus_kl",run_tag="main"):
    seed_everything(CONFIG["seed"]+fold)
    model=build_model()
    opt=torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG["learning_rate"],
        betas=(CONFIG["adam_beta1"],CONFIG["adam_beta2"]),
        weight_decay=CONFIG["weight_decay"],
    )
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=CONFIG["max_epochs"],eta_min=CONFIG["eta_min"])
    tr_loader=make_loader(train_df,True); va_loader=make_loader(val_df,False)

    C=CONFIG["prior_weight_C"] if prior_C is None else float(prior_C)
    lkl=CONFIG["lambda_kl"] if lambda_kl is None else float(lambda_kl)

    best=float("inf"); stale=0; hist=[]
    ckpt=CHECKPOINT_ROOT/f"B3_{run_tag}_fold_{fold}_best.pt"

    for epoch in range(1,CONFIG["max_epochs"]+1):
        model.train(); totals={"loss_total":0.0,"loss_ece":0.0,"loss_kl":0.0}; seen=0
        for b in tqdm(tr_loader,desc=f"B3 {run_tag} fold {fold} epoch {epoch}",leave=False):
            x=b["image"].to(DEVICE); y=b["label"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss,comp=vanilla_edl_loss(model(x),y,C,lkl,loss_mode)
            loss.backward(); opt.step()
            bs=len(y); seen+=bs
            for k in totals: totals[k]+=comp[k]*bs

        vr=edl_predict(model,va_loader,C)
        vm=metrics(vr["y_true"],vr["probs"])
        row={
            "fold":fold,"epoch":epoch,"prior_C":C,"lambda_kl":lkl,"loss_mode":loss_mode,
            "lr":opt.param_groups[0]["lr"],
            **{k:v/max(seen,1) for k,v in totals.items()},
            **{f"val_{k}":v for k,v in vm.items()}
        }
        hist.append(row)
        vb=vm["brier_score"]
        print(f"fold={fold} epoch={epoch} loss={row['loss_total']:.4f} val_Brier={vb:.4f} val_F1={vm['f1_macro']:.4f}")
        if vb<best-1e-8:
            best=vb; stale=0
            torch.save({
                "config":CONFIG,"fold":fold,"prior_C":C,"lambda_kl":lkl,
                "loss_mode":loss_mode,"epoch":epoch,"best_val_brier":best,
                "model_state_dict":model.state_dict()
            },ckpt)
        else:
            stale+=1
        sch.step()
        if stale>=CONFIG["early_stopping_patience"]:
            break

    pd.DataFrame(hist).to_csv(LOG_ROOT/f"B3_{run_tag}_fold_{fold}_history.csv",index=False)
    state=torch.load(ckpt,map_location=DEVICE)
    model.load_state_dict(state["model_state_dict"])
    model.eval()
    return model,pd.DataFrame(hist),ckpt

In [ ]:
def rejection_curve(y,p,u):
    y=np.asarray(y); pred=np.asarray(p).argmax(1); order=np.argsort(u); n=len(y)
    return pd.DataFrame([
        {"coverage":r/n,"accuracy":float(accuracy_score(y[order[:r]],pred[order[:r]]))}
        for r in range(1,n+1)
    ])

def random_rejection(y,p,repeats=50,seed=42):
    y=np.asarray(y); pred=np.asarray(p).argmax(1); n=len(y); rng=np.random.RandomState(seed)
    curves=[]
    for _ in range(repeats):
        order=rng.permutation(n)
        curves.append([accuracy_score(y[order[:r]],pred[order[:r]]) for r in range(1,n+1)])
    a=np.asarray(curves)
    return pd.DataFrame({"coverage":np.arange(1,n+1)/n,"accuracy":a.mean(0),"accuracy_sd":a.std(0)})

def curve_auc(df):
    s=df.sort_values("coverage")
    return float(np.trapz(s.accuracy,s.coverage))

def efficiency(model,loader):
    params=sum(p.numel() for p in model.parameters())
    dummy=torch.randn(1,3,224,224,device=DEVICE)
    try: flops=float(profile(model,inputs=(dummy,),verbose=False)[0])
    except Exception: flops=np.nan
    one=DataLoader(loader.dataset,batch_size=1,shuffle=False,num_workers=0)
    times=[]
    for j,b in enumerate(one):
        if j>=min(30,len(loader.dataset)): break
        x=b["image"].to(DEVICE)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t=time.perf_counter()
        with torch.no_grad(): _=logits_to_dirichlet(model(x))
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times.append((time.perf_counter()-t)*1000)
    return {
        "total_parameters":int(params),
        "single_pass_flops":flops,
        "single_pass_latency_ms_per_image":float(np.mean(times)) if times else np.nan,
    }

In [ ]:
class SaliencyEngine:
    def __init__(self,model):
        self.model=model
        self.layer=model.backbone.layer4[-1].conv3
        self.a=None; self.g=None
        self.fh=self.layer.register_forward_hook(lambda m,i,o:setattr(self,"a",o))
        self.bh=self.layer.register_full_backward_hook(lambda m,gi,go:setattr(self,"g",go[0]))
    def remove(self):
        self.fh.remove(); self.bh.remove()
    def _heat(self,scalar):
        self.model.zero_grad(set_to_none=True)
        scalar.backward()
        w=self.g.mean((2,3),keepdim=True)
        h=F.relu((w*self.a).sum(1,keepdim=True))
        h=F.interpolate(h,size=(224,224),mode="bilinear",align_corners=False)[0,0].detach().cpu().numpy()
        h-=h.min()
        if h.max()>0: h/=h.max()
        return h
    def gradcam(self,x):
        self.model.eval()
        logits=self.model(x)
        d=logits_to_dirichlet(logits)
        c=int(d["probs"].argmax(1).item())
        return self._heat(logits[:,c].sum()),c
    def esm_vacuity(self,x,C=None):
        self.model.eval()
        d=logits_to_dirichlet(self.model(x),C)
        return self._heat(d["vacuity"].sum())
    def esm_aleatoric(self,x,C=None):
        self.model.eval()
        d=logits_to_dirichlet(self.model(x),C)
        return self._heat(d["aleatoric_entropy"].sum())

In [ ]:
def save_saliency(model,test_df,fold,C=None):
    ds=LIDCDataset(test_df)
    eng=SaliencyEngine(model)
    rows=[]
    try:
        for i in tqdm(range(min(len(ds),CONFIG["esm_max_examples_per_fold"])),desc=f"Saliency fold {fold}",leave=False):
            s=ds[i]; x=s["image"].unsqueeze(0).to(DEVICE)
            gc,pred=eng.gradcam(x); ev=eng.esm_vacuity(x,C); ea=eng.esm_aleatoric(x,C)
            stem=Path(s["image_path"]).stem
            gd=GRADCAM_ROOT/f"fold_{fold}"; ed=ESM_ROOT/f"fold_{fold}"
            gd.mkdir(parents=True,exist_ok=True); ed.mkdir(parents=True,exist_ok=True)
            np.save(gd/f"{stem}_gradcam.npy",gc)
            np.save(ed/f"{stem}_esm_vacuity.npy",ev)
            np.save(ed/f"{stem}_esm_aleatoric.npy",ea)
            rows.append({"fold":fold,"image_path":s["image_path"],"patient_id":s["patient_id"],"predicted_label":pred})
    finally:
        eng.remove()
    pd.DataFrame(rows).to_csv(ESM_ROOT/f"fold_{fold}_saliency_manifest.csv",index=False)

In [ ]:
def evaluate_fold(model,test_df,fold,C=None,tag="main"):
    loader=make_loader(test_df,False)
    r=edl_predict(model,loader,C)
    p=r["probs"]; y=r["y_true"]; pred=p.argmax(1)

    pred_df=pd.DataFrame({
        "fold":fold,
        "image_path":r["paths"],
        "patient_id":r["patients"],
        "true_label":y,
        "predicted_label":pred,
        "p_benign":p[:,0],
        "p_indeterminate":p[:,1],
        "p_malignant":p[:,2],
        "vacuity":r["vacuity"],
        "aleatoric_entropy":r["aleatoric_entropy"],
        "normalized_aleatoric_entropy":r["normalized_aleatoric_entropy"],
        "dirichlet_strength":r["strength"],
    })
    pred_df.to_csv(PREDICTION_ROOT/f"B3_{tag}_fold_{fold}_predictions.csv",index=False)

    m=metrics(y,p)
    rd=rejection_curve(y,p,r["vacuity"])
    rr=random_rejection(y,p,CONFIG["random_rejection_repeats"],CONFIG["seed"]+fold)
    rd.to_csv(METRICS_ROOT/f"B3_{tag}_fold_{fold}_rejection_vacuity.csv",index=False)
    rr.to_csv(METRICS_ROOT/f"B3_{tag}_fold_{fold}_rejection_random.csv",index=False)

    row={
        "fold":fold,
        "prior_C":CONFIG["prior_weight_C"] if C is None else float(C),
        "test_patches":len(test_df),
        "test_patients":test_df.patient_id.nunique(),
        **m,
        "rejection_auc_vacuity":curve_auc(rd),
        "rejection_auc_random":curve_auc(rr),
        **efficiency(model,loader),
        "mean_vacuity":float(np.mean(r["vacuity"])),
        "mean_aleatoric_entropy":float(np.mean(r["aleatoric_entropy"])),
        "mean_dirichlet_strength":float(np.mean(r["strength"])),
    }

    np.save(METRICS_ROOT/f"B3_{tag}_fold_{fold}_confusion_matrix.npy",
            confusion_matrix(y,pred,labels=[0,1,2]))
    with open(METRICS_ROOT/f"B3_{tag}_fold_{fold}_classification_report.json","w") as f:
        json.dump(classification_report(y,pred,labels=[0,1,2],
            target_names=CONFIG["class_names"],output_dict=True,zero_division=0),f,indent=2)

    save_saliency(model,test_df,fold,C)
    pd.DataFrame([row]).to_csv(METRICS_ROOT/f"B3_{tag}_fold_{fold}_metrics.csv",index=False)
    return row

In [ ]:
RUN_ALL_FOLDS=True
all_rows=[]

if RUN_ALL_FOLDS:
    for fold in range(CONFIG["n_splits"]):
        tr=fold_frames[fold]["train"]; va=fold_frames[fold]["val"]; te=fold_frames[fold]["test"]
        model,_,_=train_one_fold(
            fold,tr,va,
            prior_C=CONFIG["prior_weight_C"],
            lambda_kl=CONFIG["lambda_kl"],
            loss_mode="ece_plus_kl",
            run_tag="main",
        )
        all_rows.append(evaluate_fold(model,te,fold,CONFIG["prior_weight_C"],"main"))
        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    fold_metrics=pd.DataFrame(all_rows)
    fold_metrics.to_csv(METRICS_ROOT/"B3_fold_metrics.csv",index=False)
    display(fold_metrics)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 172MB/s]


B3 main fold 0 epoch 1:   0%|          | 0/3 [00:00<?, ?it/s]

fold=0 epoch=1 loss=1.5266 val_Brier=0.6539 val_F1=0.2222


B3 main fold 0 epoch 2:   0%|          | 0/3 [00:00<?, ?it/s]

...

fold=4 epoch=25 loss=0.2906 val_Brier=0.5529 val_F1=0.3175


B3 main fold 4 epoch 26:   0%|          | 0/2 [00:00<?, ?it/s]

fold=4 epoch=26 loss=0.2695 val_Brier=0.5513 val_F1=0.3175


Saliency fold 4:   0%|          | 0/18 [00:00<?, ?it/s]

,fold,prior_C,test_patches,test_patients,accuracy,f1_macro,auroc_macro_ovr,brier_score,ece_15,nll,rejection_auc_vacuity,rejection_auc_random,total_parameters,single_pass_flops,single_pass_latency_ms_per_image,mean_vacuity,mean_aleatoric_entropy,mean_dirichlet_strength
0,0,1.0,19,6,0.421053,0.205128,0.366973,0.661860,0.073269,1.091432,0.572575,0.417435,23514179,4.131701e+09,10.148492,0.598179,1.097720,5.017793
1,1,1.0,22,9,0.590909,0.247619,0.506477,0.636540,0.286878,1.054226,0.721727,0.583101,23514179,4.131701e+09,7.623312,0.597009,1.093797,5.026466
2,2,1.0,23,9,0.521739,0.228571,0.358842,0.676994,0.265140,1.113145,0.368326,0.511131,23514179,4.131701e+09,10.079228,0.507223,0.972923,6.030213
3,3,1.0,12,7,0.416667,0.196078,0.712511,0.658789,0.214711,1.085571,0.504711,0.353352,23514179,4.131701e+09,8.333518,0.506483,0.947356,6.011137
4,4,1.0,18,7,0.722222,0.419355,0.376923,0.550787,0.258535,0.938397,0.664389,0.687720,23514179,4.131701e+09,11.816017,0.562519,1.037077,5.357650


In [ ]:
if RUN_ALL_FOLDS and all_rows:
    df=pd.DataFrame(all_rows)
    summary=[]
    for c in df.columns:
        if c!="fold" and pd.api.types.is_numeric_dtype(df[c]):
            v=pd.to_numeric(df[c],errors="coerce")
            summary.append({
                "metric":c,
                "mean":float(v.mean()) if v.notna().any() else np.nan,
                "std":float(v.std(ddof=1)) if v.notna().sum()>1 else np.nan,
                "n_valid_folds":int(v.notna().sum()),
            })
    summary=pd.DataFrame(summary)
    summary.to_csv(METRICS_ROOT/"B3_5fold_mean_std.csv",index=False)
    display(summary)

,metric,mean,std,n_valid_folds
0,prior_C,1.000000e+00,0.000000,5
1,test_patches,1.880000e+01,4.324350,5
2,test_patients,7.600000e+00,1.341641,5
3,accuracy,5.345179e-01,0.127808,5
4,f1_macro,2.593504e-01,0.091692,5
5,auroc_macro_ovr,4.643454e-01,0.151341,5
6,brier_score,6.369940e-01,0.050312,5
7,ece_15,2.197066e-01,0.085955,5
8,nll,1.056554e+00,0.069332,5
9,rejection_auc_vacuity,5.663456e-01,0.138569,5


In [ ]:
if RUN_ALL_FOLDS and all_rows:
    mean_f1=float(pd.DataFrame(all_rows)["f1_macro"].mean())
    milestone={
        "configuration":"B3",
        "milestone":"M2",
        "threshold_f1_macro":0.50,
        "observed_mean_f1_macro":mean_f1,
        "passed":bool(mean_f1>0.50),
        "note":"Pilot result while using approximately 20 patients."
    }
    with open(REPORT_ROOT/"B3_M2_milestone.json","w") as f:
        json.dump(milestone,f,indent=2)
    print(milestone)

{'configuration': 'B3', 'milestone': 'M2', 'threshold_f1_macro': 0.5, 'observed_mean_f1_macro': 0.2593503902801816, 'passed': False, 'note': 'Pilot result while using approximately 20 patients.'}


In [ ]:
RUN_PRIOR_C_ABLATION=False
RUN_LOSS_ABLATION=False

def run_prior_C_ablation():
    rows=[]
    fold=0
    tr,va,te=fold_frames[0]["train"],fold_frames[0]["val"],fold_frames[0]["test"]
    for C in CONFIG["ablation_prior_C_values"]:
        tag=f"ablation_C_{str(C).replace('.','p')}"
        model,_,_=train_one_fold(0,tr,va,C,CONFIG["lambda_kl"],"ece_plus_kl",tag)
        r=evaluate_fold(model,te,0,C,tag)
        r["ablation"]="prior_C"; r["value"]=C
        rows.append(r)
        del model; gc.collect()
    out=pd.DataFrame(rows)
    out.to_csv(ABLATION_ROOT/"B3_fold0_prior_C_ablation.csv",index=False)
    return out

def run_loss_ablation():
    rows=[]
    tr,va,te=fold_frames[0]["train"],fold_frames[0]["val"],fold_frames[0]["test"]
    for mode in CONFIG["ablation_loss_modes"]:
        tag=f"ablation_{mode}"
        model,_,_=train_one_fold(0,tr,va,CONFIG["prior_weight_C"],CONFIG["lambda_kl"],mode,tag)
        r=evaluate_fold(model,te,0,CONFIG["prior_weight_C"],tag)
        r["ablation"]="loss_component"; r["value"]=mode
        rows.append(r)
        del model; gc.collect()
    out=pd.DataFrame(rows)
    out.to_csv(ABLATION_ROOT/"B3_fold0_loss_isolation.csv",index=False)
    return out

if RUN_PRIOR_C_ABLATION:
    display(run_prior_C_ablation())
if RUN_LOSS_ABLATION:
    display(run_loss_ablation())

In [ ]:
saliency_ablation=pd.DataFrame([
    {"target":"Grad-CAM","scalar":"predicted class logit","gradient":"dy_c/dA"},
    {"target":"ESM-vacuity","scalar":"u=K*C/S","gradient":"du/dA"},
    {"target":"ESM-aleatoric","scalar":"H_al=-sum p log p","gradient":"dH_al/dA"},
])
saliency_ablation.to_csv(ABLATION_ROOT/"B3_saliency_target_ablation_manifest.csv",index=False)
display(saliency_ablation)

,target,scalar,gradient
0,Grad-CAM,predicted class logit,dy_c/dA
1,ESM-vacuity,u=K*C/S,du/dA
2,ESM-aleatoric,H_al=-sum p log p,dH_al/dA


In [ ]:
def paired_wilcoxon_comparison(b3_csv,other_csv,metric,other_name="other_config"):
    a=pd.read_csv(b3_csv)[["fold",metric]].rename(columns={metric:"B3"})
    b=pd.read_csv(other_csv)[["fold",metric]].rename(columns={metric:other_name})
    paired=a.merge(b,on="fold",how="inner").dropna()
    if len(paired)!=5:
        raise ValueError(f"Expected 5 paired folds; found {len(paired)}")
    stat,p=wilcoxon(paired["B3"],paired[other_name],alternative="two-sided",mode="auto")
    diff=paired["B3"].values-paired[other_name].values
    return {
        "metric":metric,"n_pairs":5,
        "wilcoxon_statistic":float(stat),"p_value":float(p),"alpha":0.05,
        "B3_mean":float(paired["B3"].mean()),
        f"{other_name}_mean":float(paired[other_name].mean()),
        "absolute_mean_difference":float(abs(np.mean(diff))),
    }

In [ ]:
manifest={
    "configuration":"B3",
    "backbone":"ResNet50",
    "uq_method":"Vanilla EDL",
    "prior_C":CONFIG["prior_weight_C"],
    "lambda_KL":CONFIG["lambda_kl"],
    "lambda_KL_note":"Proposal names lambda_KL but does not specify its numeric value; confirm before final full-data run.",
    "loss":"L_ECE + lambda_KL * L_KL",
    "xai":["Grad-CAM","ESM-vacuity","ESM-aleatoric"],
    "patients_used":int(metadata.patient_id.nunique()),
    "patches_used":int(len(metadata)),
    "folds":5,
    "seed":42,
    "ablation_support":{
        "prior_C":[0.5,1.0,2.0],
        "loss_isolation":["ece_only","ece_plus_kl"],
        "saliency_targets":["Grad-CAM","ESM-vacuity","ESM-aleatoric"],
        "lambda_max":"not applicable to B3",
        "Twarmup":"not applicable to B3",
    }
}
with open(REPORT_ROOT/"B3_experiment_manifest.json","w") as f:
    json.dump(manifest,f,indent=2)
print(json.dumps(manifest,indent=2))

{
  "configuration": "B3",
  "backbone": "ResNet50",
  "uq_method": "Vanilla EDL",
  "prior_C": 1.0,
  "lambda_KL": 1.0,
  "lambda_KL_note": "Proposal names lambda_KL but does not specify its numeric value; confirm before final full-data run.",
  "loss": "L_ECE + lambda_KL * L_KL",
  "xai": [
    "Grad-CAM",
    "ESM-vacuity",
    "ESM-aleatoric"
  ],
  "patients_used": 38,
  "patches_used": 94,
  "folds": 5,
  "seed": 42,
  "ablation_support": {
    "prior_C": [
      0.5,
      1.0,
      2.0
    ],
    "loss_isolation": [
      "ece_only",
      "ece_plus_kl"
    ],
    "saliency_targets": [
      "Grad-CAM",
      "ESM-vacuity",
      "ESM-aleatoric"
    ],
    "lambda_max": "not applicable to B3",
    "Twarmup": "not applicable to B3"
  }
}
